### Phase-1: Content-Based Filtering

### Load and Prepare the Data

In [1]:
import pandas as pd

# Load ratings
ratings = pd.read_csv("data/u.data", sep='\t', names=['user_id','movie_id', 'rating','timestamp'])

# Load movie metadata
movies = pd.read_csv("data/u.item", sep='|', encoding='latin-1', names=['movie_id','title','release_date','video_release_date','IMDB_URL','unknown','Action','Adventure','Animation',"Children's",'Comedy','Crime','Documentry','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western'])

# Merge
df = pd.merge(ratings, movies[['movie_id','title']], on='movie_id')
df.head()

,user_id,movie_id,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


### Create a Pivot table

In [3]:
user_movie_matrix = df.pivot_table(index='user_id', columns='title',values='rating')
user_movie_matrix.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,2.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,4.0,NaN


### Build Content-Based Recommender

In [11]:
# Example: Find similar movies to "Star Wars (1977)"
target_movie = 'Star Wars (1977)'

# Get ratings of the target movie
movie_ratings = user_movie_matrix[target_movie]

# Compute correlations
similar_movies = user_movie_matrix.corrwith(movie_ratings)
corr_df = pd.DataFrame(similar_movies, columns=['correlation'])
corr_df.dropna(inplace=True)

# Add number of ratings
rating_counts = df.groupby('title')['rating'].count()
corr_df['num_ratings'] = rating_counts

# Filter out movies with fewer ratings
recommendations = corr_df[corr_df['num_ratings'] > 100].sort_values('correlation', ascending=False)
recommendations.head(10)


C:\Users\sudheer kumar\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\sudheer kumar\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\sudheer kumar\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
C:\Users\sudheer kumar\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\sudheer kumar\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


,correlation,num_ratings
title,,
Star Wars (1977),1.000000,583
"Empire Strikes Back, The (1980)",0.747981,367
Return of the Jedi (1983),0.672556,507
Raiders of the Lost Ark (1981),0.536117,420
Austin Powers: International Man of Mystery (1997),0.377433,130
"Sting, The (1973)",0.367538,241
Indiana Jones and the Last Crusade (1989),0.350107,331
Pinocchio (1940),0.347868,101
"Frighteners, The (1996)",0.332729,115


### Phase-2: Collaborative Filtering (User-User or Item-Item)

### Fill Missing Values

In [13]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

### Use Cosine Similarity (Item-based Filtering)

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

# Transpose to get item-item similarity
item_similarity = cosine_similarity(user_movie_matrix_filled.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)

# Recommend similar movies
def recommend_movies(movie_name, n=10):
    similar_scores = item_similarity_df[movie_name].sort_values(ascending=False)[1:n+1]
    return similar_scores

recommend_movies('Star Wars (1977)', 5)


title
Return of the Jedi (1983)          0.884476
Raiders of the Lost Ark (1981)     0.764885
Empire Strikes Back, The (1980)    0.749819
Toy Story (1995)                   0.734572
Godfather, The (1972)              0.697332
Name: Star Wars (1977), dtype: float64

### PHASE 3: Build Flask API to Serve Recommendations